In [1]:
from langchain.agents import initialize_agent, AgentType
from langchain.agents import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults
from langchain.agents import ZeroShotAgent, AgentExecutor
from langchain.chains import LLMChain


from typing import List

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os

from dotenv import load_dotenv
load_dotenv()

True

In [4]:
search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")


'The White House, official residence of the president of the United States, in July 2008 The president of the United States is the head of state and head of government of the United States, [1] indirectly elected to a four-year term via the Electoral College. [2] The officeholder leads the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces ... Click on a president below to learn more about each presidency through an interactive timeline. The table below the graphic provides a list of presidents of the United States, their birthplaces, political parties, and terms of office. Here is a list of the presidents and vice presidents of the United States along with their parties and dates in office. Barack Obama, the 44th President of the United States, broke barriers as the first African-American president and implemented significant healthcare reforms during his tenure. The most common first name for a U.S. president is James, followed 

In [11]:
import os
import datetime

from dotenv import load_dotenv

from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.tools import DuckDuckGoSearchResults

load_dotenv()

# Configurar o modelo
llm = OpenAI(model='gpt-3.5.-turbo', api_key=os.getenv('OPENAI_API_KEY'))

# Criar um template de prompt
prompt_template = """
You are a News Summarizer. Your role is to:
1. Provide a list of the most recent news, weather, and events in a specific city-state
2. Provide a summary of the information
3. Provide a list of the sources of the information

Please return the information in the JSON format informing the city, state, country, and a list of news with their titles and summaries.

City-State: {city_state}
Search Results: {search_results}
"""

prompt = PromptTemplate(input_variables=["city_state", "search_results"], 
                        template=prompt_template)

print(prompt)

# Criar uma cadeia de LLM
chain = LLMChain(llm=llm, prompt=prompt)

# Função para buscar informações de uma cidade-estado específica usando DuckDuckGo
def fetch_city_state_info(city_state):
    date = datetime.datetime.now().strftime("%d-%m-%Y")
    
    search_tool = DuckDuckGoSearchResults()
    search_results = search_tool.run(f"{city_state} {date} news")
    
    # Processar os resultados da pesquisa para obter as informações necessárias
    query = prompt.format(city_state=city_state, search_results=search_results)
    
    print(query)

    result = chain.run(query)
    return result

# Exemplo de uso
city_state_input = "Campina Grande, PB"
info = fetch_city_state_info(city_state_input)
print(info)

input_variables=['city_state', 'search_results'] input_types={} partial_variables={} template='\nYou are a News Summarizer. Your role is to:\n1. Provide a list of the most recent news, weather, and events in a specific city-state\n2. Provide a summary of the information\n3. Provide a list of the sources of the information\n\nPlease return the information in the JSON format informing the city, state, country, and a list of news with their titles and summaries.\n\nCity-State: {city_state}\nSearch Results: {search_results}\n'

You are a News Summarizer. Your role is to:
1. Provide a list of the most recent news, weather, and events in a specific city-state
2. Provide a summary of the information
3. Provide a list of the sources of the information

Please return the information in the JSON format informing the city, state, country, and a list of news with their titles and summaries.

City-State: Campina Grande, PB
Search Results: snippet: Por g1 PB . 22/01/2025 17h41 Atualizado 22/01/2025 

ValueError: Missing some input keys: {'search_results'}

In [13]:
# Define the LLM (you can replace with a model of your choice)
llm = ChatOpenAI(
            temperature=0.0,
            model_name="gpt-3.5-turbo",  # Updated model
            openai_api_key=os.getenv('OPENAI_API_KEY')
        )

# Define the search tool (for simplicity, we use DuckDuckGo search via Langchain)
search_tool = Tool(
        name="web_search",
        func=DuckDuckGoSearchResults().run,
        description="Use this tool to search the web for city-specific information"
    )


# Create the agent
agent = initialize_agent([search_tool], agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, llm=llm, verbose=True)




# def create_agent(tools: List[Tool], prefix: str, suffix: str):
#     """Helper function to create an agent with specified tools and prompt"""
#     prompt = ZeroShotAgent.create_prompt(
#         tools,
#         prefix=prefix,
#         suffix=suffix,
#         input_variables=["input", "agent_scratchpad"]
#     )
    
#     llm_chain = LLMChain(llm, prompt=prompt)
    
#     agent = ZeroShotAgent(
#         llm_chain=llm_chain,
#         tools=tools,
#         verbose=True
#     )
    
#     return AgentExecutor.from_agent_and_tools(
#         agent=agent,
#         tools=tools,
#         verbose=True
#     )


# def create_agent_executor():
#     """Creates an agent specialized in subject matter expertise"""
#     # Define the search tool (for simplicity, we use DuckDuckGo search via Langchain)
#     search_tool = Tool(
#         name="web_search",
#         func=DuckDuckGoSearchResults().run,
#         description="Use this tool to search the web for city-specific information"
#     )

#     tools = [search_tool]
    
#     prefix = """You are a News Summarizer. Your role is to:
#     1. Provide a list of the most recent news, weather, and events in a specific city-state
#     2. Provide a summary of the information
#     3. Provide a list of the sources of the information
    
#     You have access to the following tools:"""
    
#     suffix = """City: {input}
#     {agent_scratchpad}"""

#     return create_agent(tools, prefix, suffix)


# agent = create_agent_executor()


# Function to fetch information for a specific city-state
def fetch_city_state_info(city_state):
    
    # Run the agent with the input city-state
    query = f"""You are a News Summarizer. Your role is to:
    
    1. Provide a list of the most recent news, weather, and events in a specific city-state
    2. Provide a summary of the information
    3. Provide a list of the sources of the information 

    Return a JSON object with the information for the following city-state:
    
    {city_state}"""
    

    result = agent.run(query)
    
    return result

# Example Usage
city_state_input = "Campina Grande, PB"
info = fetch_city_state_info(city_state_input)
print(info)




> Entering new AgentExecutor chain...
I need to search the web for the most recent news, weather, and events in Campina Grande, PB.
Action: web_search
Action Input: 'Campina Grande, PB news, weather, events'
Observation: snippet: Campina Grande, PB - Weather forecast from Theweather.com. Weather conditions with updates on temperature, humidity, wind speed, snow, pressure, etc. for Campina Grande, Paraíba New York New York State, title: Campina Grande, PB Weather 14 days - Meteored, link: https://www.theweather.com/campina-grande.htm, snippet: Everything you need to know about today's weather in Campina Grande, Paraíba, Brazil. High/Low, Precipitation Chances, Sunrise/Sunset, and today's Temperature History., title: Weather Today for Campina Grande, Paraíba, Brazil | AccuWeather, link: https://www.accuweather.com/en/br/campina-grande/34634/weather-today/34634, snippet: Get Campina Grande, PB, BR current weather report with temperature, feels like, wind, humidity, pressure, UV and more